
# Intelligent Hybrid Recommendation System

## Profile-Based Matching Algorithm

An intelligent hybrid recommendation system that combines Natural Language Processing (NLP), personality compatibility, and adaptive learning to generate personalized profile recommendations.

### Key Features

- NLP-based semantic analysis using Sentence Transformers
- MBTI personality compatibility matching
- Location-based compatibility scoring
- Weighted hybrid compatibility calculation
- Adaptive feedback-based weight optimization
- Top-5 personalized profile recommendations
- Interactive Streamlit application


In [4]:
%pip install pandas numpy scikit-learn matplotlib seaborn nltk sentence-transformers streamlit joblib


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip



  Using cached sentence_transformers-6.3.0-py3-none-any.whl.metadata (20 kB)
Using cached sentence_transformers-6.3.0-py3-none-any.whl (780 kB)


In [2]:
import pandas as pd
import numpy as np
import sklearn
import matplotlib
import seaborn as sns
import nltk
import sentence_transformers
import streamlit
import joblib

print("Project 1 environment is ready.")

c:\Users\ariya\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project 1 environment is ready.


In [3]:

from pathlib import Path

# Detect the project root directory
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

# Required project directories
folders = [
    "data",
    "src",
    "models",
    "app",
    "outputs"
]

for folder in folders:
    (PROJECT_ROOT / folder).mkdir(parents=True, exist_ok=True)

print("Project workspace created successfully.")
print("Project root:", PROJECT_ROOT)


Project workspace created successfully.
Project root: c:\PROJECTS\intelligent-hybrid-recommendation-system


In [ ]:

# ============================================
# INTELLIGENT HYBRID RECOMMENDATION SYSTEM
# DATASET SPECIFICATION
# ============================================

USER_COLUMNS = [
    "user_id",
    "name",
    "age",
    "location",
    "profession",
    "experience_years",
    "professional_summary",
    "about_me",
    "mbti",
    "interests"
]

FEEDBACK_COLUMNS = [
    "user_id",
    "matched_user_id",
    "action",
    "timestamp"
]

ALLOWED_MBTI = [
    "INTJ", "INTP", "ENTJ", "ENTP",
    "INFJ", "INFP", "ENFJ", "ENFP",
    "ISTJ", "ISFJ", "ESTJ", "ESFJ",
    "ISTP", "ISFP", "ESTP", "ESFP"
]

print("USER COLUMNS:")
print(USER_COLUMNS)

print("\nFEEDBACK COLUMNS:")
print(FEEDBACK_COLUMNS)

print("\nALLOWED MBTI TYPES:")
print(ALLOWED_MBTI)


In [ ]:
NUM_USERS = 100
INTERACTIONS_PER_USER = 8

print("Users:", NUM_USERS)
print("Interactions per user:", INTERACTIONS_PER_USER)
print("Expected feedback rows:", NUM_USERS * INTERACTIONS_PER_USER)

In [ ]:
# ============================================
# UNLOX PROJECT 1
# PROFILE DATA CONFIGURATION
# ============================================

PROFILE_CONFIG = {
    "locations": [
        "Bangalore",
        "Hyderabad",
        "Pune",
        "Mumbai",
        "Delhi",
        "Chennai",
        "Kolkata",
        "Bhubaneswar",
        "Ahmedabad",
        "Jaipur"
    ],

    "professions": [
        "Data Scientist",
        "Machine Learning Engineer",
        "Software Engineer",
        "Data Analyst",
        "AI Engineer",
        "Product Manager",
        "Business Analyst",
        "Cybersecurity Analyst",
        "Cloud Engineer",
        "UX Designer",
        "Research Scientist",
        "Financial Analyst",
        "Marketing Analyst",
        "HR Specialist",
        "DevOps Engineer"
    ],

    "industries": [
        "Healthcare",
        "FinTech",
        "E-commerce",
        "Education",
        "Technology",
        "Banking",
        "Automotive",
        "Consulting",
        "Telecommunications",
        "Manufacturing"
    ],

    "experience_levels": [
        1, 2, 3, 4, 5, 6, 7, 8, 9, 10
    ]
}

print("Profile configuration created.")
print("Locations:", len(PROFILE_CONFIG["locations"]))
print("Professions:", len(PROFILE_CONFIG["professions"]))
print("Industries:", len(PROFILE_CONFIG["industries"]))

In [ ]:
FIRST_NAMES = [
    "Aarav", "Aditya", "Akash", "Aman", "Ananya",
    "Arjun", "Aryan", "Avni", "Bhavna", "Dev",
    "Diya", "Ishaan", "Isha", "Kabir", "Karan",
    "Kavya", "Kiara", "Manav", "Meera", "Mihir",
    "Nikhil", "Nisha", "Pooja", "Pranav", "Rahul",
    "Riya", "Rohan", "Sahil", "Sanya", "Shreya",
    "Simran", "Sneha", "Tanmay", "Tanya", "Varun",
    "Ved", "Vikram", "Yash", "Zoya", "Anika",
    "Aniket", "Anjali", "Harsh", "Neha", "Rajat",
    "Ritika", "Siddharth", "Sonia", "Vivek", "Aditi"
]

LAST_NAMES = [
    "Sharma", "Patel", "Singh", "Kumar", "Mohanty",
    "Das", "Mishra", "Nayak", "Mehta", "Gupta",
    "Joshi", "Reddy", "Verma", "Sahu", "Jena",
    "Roy", "Bose", "Panda", "Rao", "Nair"
]

print("First names:", len(FIRST_NAMES))
print("Last names:", len(LAST_NAMES))

In [ ]:
CAREER_PROFILES = {
    "Data Scientist": {
        "skills": ["Python", "SQL", "statistics", "machine learning", "data visualization"],
        "industries": ["Healthcare", "FinTech", "E-commerce", "Technology"],
        "goals": [
            "building predictive analytics solutions",
            "developing data-driven decision systems",
            "applying machine learning to real-world problems"
        ]
    },

    "Machine Learning Engineer": {
        "skills": ["Python", "PyTorch", "TensorFlow", "scikit-learn", "MLOps"],
        "industries": ["Technology", "Healthcare", "FinTech", "E-commerce"],
        "goals": [
            "deploying reliable machine learning systems",
            "building scalable AI applications",
            "improving production ML pipelines"
        ]
    },

    "Software Engineer": {
        "skills": ["Python", "Java", "REST APIs", "Git", "software architecture"],
        "industries": ["Technology", "FinTech", "E-commerce", "Telecommunications"],
        "goals": [
            "building scalable software products",
            "designing reliable backend systems",
            "developing high-quality production applications"
        ]
    },

    "Data Analyst": {
        "skills": ["SQL", "Python", "Excel", "Power BI", "business reporting"],
        "industries": ["Healthcare", "Banking", "E-commerce", "Consulting"],
        "goals": [
            "turning business data into actionable insights",
            "improving reporting and decision-making",
            "supporting data-driven business strategy"
        ]
    },

    "AI Engineer": {
        "skills": ["Python", "deep learning", "NLP", "computer vision", "Transformers"],
        "industries": ["Technology", "Healthcare", "Automotive", "Education"],
        "goals": [
            "building intelligent AI-powered products",
            "developing practical generative AI solutions",
            "applying deep learning to complex problems"
        ]
    },

    "Product Manager": {
        "skills": ["product strategy", "market research", "analytics", "roadmapping", "communication"],
        "industries": ["Technology", "FinTech", "E-commerce", "Education"],
        "goals": [
            "building products that solve meaningful user problems",
            "leading data-informed product strategy",
            "creating scalable digital experiences"
        ]
    },

    "Business Analyst": {
        "skills": ["requirements analysis", "SQL", "process modeling", "Excel", "stakeholder management"],
        "industries": ["Banking", "Consulting", "Healthcare", "Manufacturing"],
        "goals": [
            "improving business processes through analytics",
            "bridging business needs with technology",
            "supporting strategic decision-making"
        ]
    },

    "Cybersecurity Analyst": {
        "skills": ["network security", "SIEM", "threat analysis", "Python", "incident response"],
        "industries": ["Banking", "Technology", "Telecommunications", "Consulting"],
        "goals": [
            "protecting organizations from evolving cyber threats",
            "improving security monitoring and response",
            "building stronger defensive security systems"
        ]
    },

    "Cloud Engineer": {
        "skills": ["AWS", "Azure", "Docker", "Kubernetes", "Terraform"],
        "industries": ["Technology", "Telecommunications", "FinTech", "E-commerce"],
        "goals": [
            "designing scalable cloud infrastructure",
            "improving cloud reliability and efficiency",
            "automating modern infrastructure operations"
        ]
    },

    "UX Designer": {
        "skills": ["Figma", "user research", "wireframing", "prototyping", "usability testing"],
        "industries": ["Technology", "E-commerce", "Education", "Healthcare"],
        "goals": [
            "creating accessible and intuitive digital experiences",
            "improving products through user-centered design",
            "designing interfaces that solve real user needs"
        ]
    }
}

print("Career profiles configured:", len(CAREER_PROFILES))

In [ ]:
# ============================================
# UNLOX PROJECT 1
# MBTI ROLE CONFIGURATION
# ============================================

ROLE_MBTI = {
    "Data Scientist": ["INTJ", "INTP", "ISTJ"],
    "Machine Learning Engineer": ["INTJ", "INTP", "ISTP"],
    "Software Engineer": ["INTJ", "INTP", "ISTJ", "ISTP"],
    "Data Analyst": ["ISTJ", "INTP", "ESTJ"],
    "AI Engineer": ["INTJ", "INTP", "ENTP"],
    "Product Manager": ["ENTJ", "ENFJ", "ESTJ"],
    "Business Analyst": ["ESTJ", "ENTJ", "ISTJ"],
    "Cybersecurity Analyst": ["ISTJ", "INTJ", "ISTP"],
    "Cloud Engineer": ["ISTJ", "INTJ", "ISTP"],
    "UX Designer": ["INFP", "ENFP", "ISFP"]
}

print("MBTI role configuration created.")
print("Roles configured:", len(ROLE_MBTI))

# Verify every assigned type is one of UNLOX's allowed types
invalid_types = {
    mbti
    for types in ROLE_MBTI.values()
    for mbti in types
    if mbti not in ALLOWED_MBTI
}

print("Invalid MBTI types:", invalid_types)

In [ ]:
PROFESSIONAL_SUMMARY_TEMPLATES = [
    "{profession} with {experience} years of experience in {industry}, skilled in {skills}. "
    "Focused on {goal} and applying practical solutions to real-world business problems.",

    "Working as a {profession} with {experience} years of experience across the {industry} sector. "
    "My core strengths include {skills}, and my career goal is {goal}.",

    "{profession} experienced in {industry}, with {experience} years of professional experience "
    "and expertise in {skills}. Interested in {goal} while creating measurable and reliable outcomes."
]

print("Professional summary templates:", len(PROFESSIONAL_SUMMARY_TEMPLATES))

In [ ]:
ABOUT_ME_TEMPLATES = [
    "I enjoy {personality_trait} and prefer a {work_style} work environment. "
    "I value {value} and spend my free time exploring {interest}.",

    "I am someone who enjoys {personality_trait} while working in a {work_style} environment. "
    "I believe in {value}, and I am interested in {interest}.",

    "My personality is shaped by {personality_trait}, and I work best in a {work_style} setting. "
    "I value {value} and enjoy learning more about {interest}."
]

PERSONALITY_TRAITS = [
    "solving challenging problems",
    "exploring new ideas",
    "helping people solve problems",
    "learning through experimentation",
    "understanding complex systems",
    "creative thinking",
    "collaborating with others",
    "organizing ideas and processes",
    "building practical solutions",
    "continuous learning"
]

WORK_STYLES = [
    "collaborative",
    "structured",
    "flexible",
    "independent",
    "fast-paced",
    "research-oriented"
]

VALUES = [
    "continuous learning",
    "honest communication",
    "teamwork",
    "creativity",
    "responsibility",
    "curiosity",
    "practical problem solving",
    "professional growth"
]

INTERESTS = [
    "technology and innovation",
    "reading and research",
    "photography",
    "travel and exploring new places",
    "fitness and outdoor activities",
    "music",
    "entrepreneurship",
    "community activities",
    "art and design",
    "artificial intelligence"
]

print("About Me templates:", len(ABOUT_ME_TEMPLATES))
print("Personality traits:", len(PERSONALITY_TRAITS))
print("Work styles:", len(WORK_STYLES))
print("Values:", len(VALUES))
print("Interests:", len(INTERESTS))

In [ ]:
PROFESSION_INTERESTS = {
    "Data Scientist": [
        "data visualization", "machine learning", "research",
        "technology", "reading", "analytics"
    ],

    "Machine Learning Engineer": [
        "artificial intelligence", "deep learning", "open source",
        "technology", "research", "robotics"
    ],

    "Software Engineer": [
        "software development", "open source", "technology",
        "gaming", "problem solving", "coding"
    ],

    "Data Analyst": [
        "analytics", "business strategy", "visualization",
        "research", "finance", "technology"
    ],

    "AI Engineer": [
        "artificial intelligence", "NLP", "computer vision",
        "robotics", "research", "technology"
    ],

    "Product Manager": [
        "product strategy", "startups", "technology",
        "business", "user experience", "innovation"
    ],

    "Business Analyst": [
        "business strategy", "analytics", "finance",
        "consulting", "problem solving", "research"
    ],

    "Cybersecurity Analyst": [
        "cybersecurity", "technology", "networking",
        "ethical hacking", "privacy", "research"
    ],

    "Cloud Engineer": [
        "cloud computing", "DevOps", "automation",
        "technology", "open source", "infrastructure"
    ],

    "UX Designer": [
        "design", "user research", "photography",
        "art", "creativity", "technology"
    ]
}

print("Profession-interest mapping created.")
print("Professions configured:", len(PROFESSION_INTERESTS))

In [ ]:
# Validate the profile-generation configuration

required_professions = set(PROFILE_CONFIG["professions"])

career_professions = set(CAREER_PROFILES.keys())
mbti_professions = set(ROLE_MBTI.keys())
interest_professions = set(PROFESSION_INTERESTS.keys())

print("Total professions required:", len(required_professions))
print("Career profiles:", len(career_professions))
print("MBTI mappings:", len(mbti_professions))
print("Interest mappings:", len(interest_professions))

print("\nMissing career profiles:",
      required_professions - career_professions)

print("Missing MBTI mappings:",
      required_professions - mbti_professions)

print("Missing interest mappings:",
      required_professions - interest_professions)

In [ ]:
# Add the missing profession configurations

CAREER_PROFILES.update({
    "Financial Analyst": {
        "skills": ["financial modeling", "Excel", "SQL", "data analysis", "forecasting"],
        "industries": ["Banking", "FinTech", "Consulting", "Manufacturing"],
        "goals": [
            "building accurate financial forecasts",
            "supporting data-driven investment decisions",
            "improving financial planning and analysis"
        ]
    },

    "Research Scientist": {
        "skills": ["research methodology", "Python", "statistics", "experimentation", "data analysis"],
        "industries": ["Healthcare", "Technology", "Education", "Automotive"],
        "goals": [
            "developing evidence-based solutions",
            "conducting meaningful applied research",
            "turning research findings into practical applications"
        ]
    },

    "HR Specialist": {
        "skills": ["talent management", "recruitment", "employee relations", "HR analytics", "communication"],
        "industries": ["Technology", "Consulting", "Manufacturing", "Healthcare"],
        "goals": [
            "building effective workplace cultures",
            "improving employee experiences",
            "using people analytics to support organizational growth"
        ]
    },

    "DevOps Engineer": {
        "skills": ["Docker", "Kubernetes", "CI/CD", "Linux", "AWS"],
        "industries": ["Technology", "FinTech", "Telecommunications", "E-commerce"],
        "goals": [
            "automating reliable software delivery",
            "improving infrastructure scalability",
            "building efficient and resilient deployment pipelines"
        ]
    },

    "Marketing Analyst": {
        "skills": ["market research", "SQL", "Google Analytics", "data visualization", "campaign analysis"],
        "industries": ["E-commerce", "Technology", "Consulting", "Retail"],
        "goals": [
            "using analytics to improve marketing decisions",
            "understanding customer behavior",
            "optimizing campaigns through data-driven insights"
        ]
    }
})

ROLE_MBTI.update({
    "Financial Analyst": ["ISTJ", "INTJ", "ESTJ"],
    "Research Scientist": ["INTP", "INTJ", "ISTP"],
    "HR Specialist": ["ENFJ", "ESFJ", "ENFP"],
    "DevOps Engineer": ["ISTP", "INTJ", "ISTJ"],
    "Marketing Analyst": ["ENTP", "ENFP", "ESTP"]
})

PROFESSION_INTERESTS.update({
    "Financial Analyst": [
        "finance", "investment", "business strategy",
        "economics", "analytics", "reading"
    ],

    "Research Scientist": [
        "scientific research", "technology", "reading",
        "experimentation", "innovation", "data analysis"
    ],

    "HR Specialist": [
        "people development", "leadership", "communication",
        "community activities", "psychology", "teamwork"
    ],

    "DevOps Engineer": [
        "cloud computing", "automation", "open source",
        "technology", "Linux", "infrastructure"
    ],

    "Marketing Analyst": [
        "consumer behavior", "marketing", "analytics",
        "business", "social media", "creative strategy"
    ]
})

print("Career profiles:", len(CAREER_PROFILES))
print("MBTI mappings:", len(ROLE_MBTI))
print("Interest mappings:", len(PROFESSION_INTERESTS))

In [ ]:
required_professions = set(PROFILE_CONFIG["professions"])

print("Missing career profiles:",
      required_professions - set(CAREER_PROFILES.keys()))

print("Missing MBTI mappings:",
      required_professions - set(ROLE_MBTI.keys()))

print("Missing interest mappings:",
      required_professions - set(PROFESSION_INTERESTS.keys()))

In [ ]:
import random
import pandas as pd

random.seed(42)

# Name pools
available_first_names = FIRST_NAMES.copy()
available_last_names = LAST_NAMES.copy()

random.shuffle(available_first_names)
random.shuffle(available_last_names)

users = []

for i in range(NUM_USERS):
    profession = random.choice(PROFILE_CONFIG["professions"])
    career = CAREER_PROFILES[profession]

    experience = random.randint(1, 10)
    age = random.randint(max(21, 21 + experience), 45)

    location = random.choice(PROFILE_CONFIG["locations"])
    industry = random.choice(career["industries"])
    skills = random.sample(
        career["skills"],
        k=min(3, len(career["skills"]))
    )

    goal = random.choice(career["goals"])

    mbti = random.choice(ROLE_MBTI[profession])

    personality_trait = random.choice(PERSONALITY_TRAITS)
    work_style = random.choice(WORK_STYLES)
    value = random.choice(VALUES)

    profession_interests = PROFESSION_INTERESTS[profession]
    interests = random.sample(
        profession_interests,
        k=min(3, len(profession_interests))
    )

    first_name = random.choice(FIRST_NAMES)
    last_name = random.choice(LAST_NAMES)

    professional_summary = random.choice(
        PROFESSIONAL_SUMMARY_TEMPLATES
    ).format(
        profession=profession,
        experience=experience,
        industry=industry,
        skills=", ".join(skills),
        goal=goal
    )

    about_me = random.choice(
        ABOUT_ME_TEMPLATES
    ).format(
        personality_trait=personality_trait,
        work_style=work_style,
        value=value,
        interest=", ".join(interests)
    )

    users.append({
        "user_id": f"U{i+1:03d}",
        "name": f"{first_name} {last_name}",
        "age": age,
        "location": location,
        "profession": profession,
        "experience_years": experience,
        "professional_summary": professional_summary,
        "about_me": about_me,
        "mbti": mbti,
        "interests": ", ".join(interests)
    })

users_df = pd.DataFrame(users)

print("Users generated:", len(users_df))
print("Columns:", list(users_df.columns))

users_df.head()

In [ ]:
# ============================================
# UNLOX PROJECT 1
# USERS DATASET VALIDATION
# ============================================

print("Number of users:", len(users_df))
print("Required columns present:", list(users_df.columns) == USER_COLUMNS)

print("\nMissing values:")
print(users_df.isnull().sum())

print("\nDuplicate rows:", users_df.duplicated().sum())
print("Duplicate User IDs:", users_df["user_id"].duplicated().sum())

print("\nInvalid MBTI values:")
print(set(users_df["mbti"]) - set(ALLOWED_MBTI))

print("\nEmpty text fields:")
print("Professional Summary:",
      (users_df["professional_summary"].str.strip() == "").sum())
print("About Me:",
      (users_df["about_me"].str.strip() == "").sum())
print("Interests:",
      (users_df["interests"].str.strip() == "").sum())

print("\nUnique professions:", users_df["profession"].nunique())
print("Unique locations:", users_df["location"].nunique())
print("Unique MBTI types:", users_df["mbti"].nunique())

In [ ]:
# ============================================
# UNLOX PROJECT 1
# TEXT QUALITY VALIDATION
# ============================================

users_df["summary_words"] = users_df["professional_summary"].str.split().str.len()
users_df["about_words"] = users_df["about_me"].str.split().str.len()

print("Professional Summary word-count range:")
print(users_df["summary_words"].min(), "to", users_df["summary_words"].max())

print("\nAbout Me word-count range:")
print(users_df["about_words"].min(), "to", users_df["about_words"].max())

print("\nSample Professional Summaries:")
print(users_df["professional_summary"].head(3).to_string(index=False))

print("\nSample About Me:")
print(users_df["about_me"].head(3).to_string(index=False))

In [ ]:
# Remove validation-only columns
users_df = users_df.drop(
    columns=["summary_words", "about_words"],
    errors="ignore"
)

# Confirm exact UNLOX schema
users_df = users_df[USER_COLUMNS]

# Save users.csv
USERS_PATH = f"{PROJECT_ROOT}/data/users.csv"
users_df.to_csv(USERS_PATH, index=False)

print("users.csv saved successfully.")
print("Path:", USERS_PATH)
print("Rows:", len(users_df))
print("Columns:", list(users_df.columns))

In [ ]:
# ============================================
# UNLOX PROJECT 1
# FEEDBACK DATA GENERATION
# ============================================

from datetime import datetime, timedelta

random.seed(42)
np.random.seed(42)

feedback_records = []

# Precompute user records
user_records = users_df.to_dict("records")

for user in user_records:

    # Select 8 different users to show as potential matches
    candidates = [
        other for other in user_records
        if other["user_id"] != user["user_id"]
    ]

    selected_matches = random.sample(
        candidates,
        INTERACTIONS_PER_USER
    )

    for matched in selected_matches:

        # Basic behavioral signals
        same_location = (
            user["location"] == matched["location"]
        )

        same_profession = (
            user["profession"] == matched["profession"]
        )

        shared_interests = len(
            set(user["interests"].split(", "))
            &
            set(matched["interests"].split(", "))
        )

        same_mbti = (
            user["mbti"] == matched["mbti"]
        )

        # Initial behavioral preference score
        preference_score = 0.20

        if same_location:
            preference_score += 0.15

        if same_profession:
            preference_score += 0.25

        if shared_interests >= 1:
            preference_score += 0.15

        if shared_interests >= 2:
            preference_score += 0.10

        if same_mbti:
            preference_score += 0.05

        # Add realistic behavioral variation
        preference_score += np.random.normal(0, 0.08)

        preference_score = np.clip(
            preference_score,
            0.05,
            0.95
        )

        action = int(
            np.random.random() < preference_score
        )

        # Generate a reproducible timestamp
        days_offset = random.randint(0, 90)

        timestamp = (
            datetime(2025, 1, 1)
            + timedelta(days=days_offset)
        ).strftime("%Y-%m-%d")

        feedback_records.append({
            "user_id": user["user_id"],
            "matched_user_id": matched["user_id"],
            "action": action,
            "timestamp": timestamp
        })

feedback_df = pd.DataFrame(feedback_records)

print("Feedback rows generated:", len(feedback_df))
print("Accepts:", int(feedback_df["action"].sum()))
print("Rejects:", int((feedback_df["action"] == 0).sum()))

In [ ]:
# ============================================
# UNLOX PROJECT 1
# FEEDBACK DATASET VALIDATION
# ============================================

valid_user_ids = set(users_df["user_id"])

print("Total feedback rows:", len(feedback_df))

# Required columns
print(
    "Required columns present:",
    list(feedback_df.columns) == FEEDBACK_COLUMNS
)

# Missing values
print("\nMissing values:")
print(feedback_df.isnull().sum())

# Valid actions
invalid_actions = set(feedback_df["action"]) - {0, 1}
print("\nInvalid actions:", invalid_actions)

# Valid user IDs
invalid_viewers = set(feedback_df["user_id"]) - valid_user_ids
invalid_matches = set(feedback_df["matched_user_id"]) - valid_user_ids

print("Invalid viewer IDs:", invalid_viewers)
print("Invalid matched IDs:", invalid_matches)

# Self matches
self_matches = (
    feedback_df["user_id"] ==
    feedback_df["matched_user_id"]
).sum()

print("Self matches:", self_matches)

# Duplicate interactions
duplicate_rows = feedback_df.duplicated().sum()
print("Duplicate rows:", duplicate_rows)

# Interactions per user
interaction_counts = feedback_df["user_id"].value_counts()

print(
    "\nInteractions per user — minimum:",
    interaction_counts.min()
)

print(
    "Interactions per user — maximum:",
    interaction_counts.max()
)

print(
    "Users with fewer than 5 interactions:",
    (interaction_counts < 5).sum()
)

print(
    "Users with more than 10 interactions:",
    (interaction_counts > 10).sum()
)

In [ ]:
FEEDBACK_PATH = f"{PROJECT_ROOT}/data/feedback.csv"

feedback_df.to_csv(
    FEEDBACK_PATH,
    index=False
)

print("feedback.csv saved successfully.")
print("Path:", FEEDBACK_PATH)
print("Rows:", len(feedback_df))
print("Columns:", list(feedback_df.columns))

In [ ]:
import nltk

nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

print("NLTK resources ready.")

In [ ]:
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

STOP_WORDS = set(stopwords.words("english"))
LEMMATIZER = WordNetLemmatizer()


def preprocess_text(text):
    """
    Clean and normalize profile text according to
    the UNLOX Project 1 NLP requirements.
    """

    text = str(text).lower()

    # Remove punctuation and non-alphabetic characters
    text = re.sub(r"[^a-z\s]", " ", text)

    # Tokenize
    tokens = text.split()

    # Remove stopwords + lemmatize
    tokens = [
        LEMMATIZER.lemmatize(word)
        for word in tokens
        if word not in STOP_WORDS
    ]

    return " ".join(tokens)


# Test the function on one real profile
sample_text = users_df.loc[0, "professional_summary"]

print("Original:")
print(sample_text)

print("\nProcessed:")
print(preprocess_text(sample_text))

In [ ]:
users_df["professional_summary_clean"] = (
    users_df["professional_summary"]
    .apply(preprocess_text)
)

users_df["about_me_clean"] = (
    users_df["about_me"]
    .apply(preprocess_text)
)

# Combine both required free-text fields
users_df["profile_text_clean"] = (
    users_df["professional_summary_clean"]
    + " "
    + users_df["about_me_clean"]
)

print("Preprocessing applied to all users.")

print("\nCleaned columns:")
print([
    "professional_summary_clean",
    "about_me_clean",
    "profile_text_clean"
])

print("\nSample cleaned profile:")
print(users_df.loc[0, "profile_text_clean"])

In [ ]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

print("Sentence Transformer model loaded successfully.")
print("Model:", EMBEDDING_MODEL_NAME)

In [ ]:
profile_embeddings = embedding_model.encode(
    users_df["profile_text_clean"].tolist(),
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Embeddings generated successfully.")
print("Number of profiles:", profile_embeddings.shape[0])
print("Embedding dimensions:", profile_embeddings.shape[1])

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Calculate pairwise semantic similarity between all users
text_similarity_matrix = cosine_similarity(profile_embeddings)

print("Cosine similarity matrix created.")
print("Matrix shape:", text_similarity_matrix.shape)

# Check similarity between the first two users
print(
    "U001 vs U002 similarity:",
    round(float(text_similarity_matrix[0, 1]), 4)
)

In [ ]:
# ============================================
# WHAT WE ARE BUILDING
# MBTI Compatibility Scoring
# ============================================

# UNLOX requires a static MBTI rule/matrix.
# Scores range from 0 to 100.

MBTI_COMPATIBILITY = {
    "INTJ": {
        "INTJ": 70, "INTP": 85, "ENTJ": 90, "ENTP": 95,
        "INFJ": 85, "INFP": 80, "ENFJ": 90, "ENFP": 100,
        "ISTJ": 75, "ISFJ": 65, "ESTJ": 80, "ESFJ": 65,
        "ISTP": 80, "ISFP": 70, "ESTP": 75, "ESFP": 65
    },

    "INTP": {
        "INTJ": 85, "INTP": 75, "ENTJ": 85, "ENTP": 95,
        "INFJ": 90, "INFP": 85, "ENFJ": 80, "ENFP": 95,
        "ISTJ": 75, "ISFJ": 65, "ESTJ": 70, "ESFJ": 60,
        "ISTP": 85, "ISFP": 75, "ESTP": 75, "ESFP": 65
    },

    "ENTJ": {
        "INTJ": 90, "INTP": 85, "ENTJ": 80, "ENTP": 90,
        "INFJ": 85, "INFP": 75, "ENFJ": 90, "ENFP": 85,
        "ISTJ": 85, "ISFJ": 70, "ESTJ": 80, "ESFJ": 70,
        "ISTP": 80, "ISFP": 65, "ESTP": 85, "ESFP": 70
    },

    "ENTP": {
        "INTJ": 95, "INTP": 95, "ENTJ": 90, "ENTP": 80,
        "INFJ": 90, "INFP": 85, "ENFJ": 90, "ENFP": 95,
        "ISTJ": 70, "ISFJ": 65, "ESTJ": 75, "ESFJ": 65,
        "ISTP": 85, "ISFP": 75, "ESTP": 90, "ESFP": 80
    },

    "INFJ": {
        "INTJ": 85, "INTP": 90, "ENTJ": 85, "ENTP": 90,
        "INFJ": 80, "INFP": 90, "ENFJ": 95, "ENFP": 90,
        "ISTJ": 75, "ISFJ": 85, "ESTJ": 65, "ESFJ": 80,
        "ISTP": 70, "ISFP": 85, "ESTP": 60, "ESFP": 75
    },

    "INFP": {
        "INTJ": 80, "INTP": 85, "ENTJ": 75, "ENTP": 85,
        "INFJ": 90, "INFP": 80, "ENFJ": 90, "ENFP": 95,
        "ISTJ": 65, "ISFJ": 85, "ESTJ": 55, "ESFJ": 80,
        "ISTP": 70, "ISFP": 90, "ESTP": 60, "ESFP": 85
    },

    "ENFJ": {
        "INTJ": 90, "INTP": 80, "ENTJ": 90, "ENTP": 90,
        "INFJ": 95, "INFP": 90, "ENFJ": 85, "ENFP": 95,
        "ISTJ": 70, "ISFJ": 85, "ESTJ": 75, "ESFJ": 90,
        "ISTP": 65, "ISFP": 85, "ESTP": 65, "ESFP": 85
    },

    "ENFP": {
        "INTJ": 100, "INTP": 95, "ENTJ": 85, "ENTP": 95,
        "INFJ": 90, "INFP": 95, "ENFJ": 95, "ENFP": 85,
        "ISTJ": 65, "ISFJ": 80, "ESTJ": 60, "ESFJ": 80,
        "ISTP": 70, "ISFP": 90, "ESTP": 70, "ESFP": 90
    },

    "ISTJ": {
        "INTJ": 75, "INTP": 75, "ENTJ": 85, "ENTP": 70,
        "INFJ": 75, "INFP": 65, "ENFJ": 70, "ENFP": 65,
        "ISTJ": 80, "ISFJ": 90, "ESTJ": 95, "ESFJ": 85,
        "ISTP": 85, "ISFP": 75, "ESTP": 85, "ESFP": 70
    },

    "ISFJ": {
        "INTJ": 65, "INTP": 65, "ENTJ": 70, "ENTP": 65,
        "INFJ": 85, "INFP": 85, "ENFJ": 85, "ENFP": 80,
        "ISTJ": 90, "ISFJ": 85, "ESTJ": 85, "ESFJ": 95,
        "ISTP": 75, "ISFP": 90, "ESTP": 70, "ESFP": 85
    },

    "ESTJ": {
        "INTJ": 80, "INTP": 70, "ENTJ": 80, "ENTP": 75,
        "INFJ": 65, "INFP": 55, "ENFJ": 75, "ENFP": 60,
        "ISTJ": 95, "ISFJ": 85, "ESTJ": 80, "ESFJ": 90,
        "ISTP": 85, "ISFP": 65, "ESTP": 90, "ESFP": 75
    },

    "ESFJ": {
        "INTJ": 65, "INTP": 60, "ENTJ": 70, "ENTP": 65,
        "INFJ": 80, "INFP": 80, "ENFJ": 90, "ENFP": 80,
        "ISTJ": 85, "ISFJ": 95, "ESTJ": 90, "ESFJ": 85,
        "ISTP": 65, "ISFP": 90, "ESTP": 75, "ESFP": 95
    },

    "ISTP": {
        "INTJ": 80, "INTP": 85, "ENTJ": 80, "ENTP": 85,
        "INFJ": 70, "INFP": 70, "ENFJ": 65, "ENFP": 70,
        "ISTJ": 85, "ISFJ": 75, "ESTJ": 85, "ESFJ": 65,
        "ISTP": 80, "ISFP": 90, "ESTP": 95, "ESFP": 85
    },

    "ISFP": {
        "INTJ": 70, "INTP": 75, "ENTJ": 65, "ENTP": 75,
        "INFJ": 85, "INFP": 90, "ENFJ": 85, "ENFP": 90,
        "ISTJ": 75, "ISFJ": 90, "ESTJ": 65, "ESFJ": 90,
        "ISTP": 90, "ISFP": 85, "ESTP": 80, "ESFP": 95
    },

    "ESTP": {
        "INTJ": 75, "INTP": 75, "ENTJ": 85, "ENTP": 90,
        "INFJ": 60, "INFP": 60, "ENFJ": 65, "ENFP": 70,
        "ISTJ": 85, "ISFJ": 70, "ESTJ": 90, "ESFJ": 75,
        "ISTP": 95, "ISFP": 80, "ESTP": 85, "ESFP": 95
    },

    "ESFP": {
        "INTJ": 65, "INTP": 65, "ENTJ": 70, "ENTP": 80,
        "INFJ": 75, "INFP": 85, "ENFJ": 85, "ENFP": 90,
        "ISTJ": 70, "ISFJ": 85, "ESTJ": 75, "ESFJ": 95,
        "ISTP": 85, "ISFP": 95, "ESTP": 95, "ESFP": 85
    }
}

# Verify the matrix contains all 16 × 16 combinations
matrix_valid = (
    set(MBTI_COMPATIBILITY.keys()) == set(ALLOWED_MBTI)
    and all(
        set(MBTI_COMPATIBILITY[mbti].keys()) == set(ALLOWED_MBTI)
        for mbti in ALLOWED_MBTI
    )
)

print("MBTI matrix valid:", matrix_valid)
print("INTJ + ENFP score:", MBTI_COMPATIBILITY["INTJ"]["ENFP"])

In [ ]:
# ============================================
# WHAT WE ARE BUILDING
# MBTI Compatibility Function
# ============================================

def get_mbti_score(mbti_a, mbti_b):
    """
    Return the rule-based MBTI compatibility score
    between two users.
    """

    if mbti_a not in ALLOWED_MBTI:
        raise ValueError(f"Invalid MBTI type: {mbti_a}")

    if mbti_b not in ALLOWED_MBTI:
        raise ValueError(f"Invalid MBTI type: {mbti_b}")

    return MBTI_COMPATIBILITY[mbti_a][mbti_b]


# Test the function
test_score = get_mbti_score("INTJ", "ENFP")

print("INTJ + ENFP compatibility:", test_score)
print("Score range:", 0, "to", 100)

In [ ]:
# ============================================
# Location Compatibility Function
# ============================================

def get_location_score(location_a, location_b):
    """
    Calculate location compatibility between two users.

    Same location:
        100
    Different location:
        40
    """

    if not location_a or not location_b:
        raise ValueError("Location cannot be empty.")

    return 100 if location_a == location_b else 40


# Test the function
same_city_score = get_location_score("Bangalore", "Bangalore")
different_city_score = get_location_score("Bangalore", "Pune")

print("Same location score:", same_city_score)
print("Different location score:", different_city_score)

In [ ]:
# ============================================
# Hybrid Compatibility Scoring
# ============================================

# Initial weights for the hybrid scoring engine
INITIAL_WEIGHTS = {
    "text": 0.50,
    "mbti": 0.30,
    "location": 0.20
}


def calculate_compatibility_score(
    text_similarity,
    mbti_score,
    location_score,
    weights=None
):
    """
    Calculate the overall compatibility score
    using the UNLOX weighted aggregation formula.
    """

    if weights is None:
        weights = INITIAL_WEIGHTS

    total_weight = (
        weights["text"]
        + weights["mbti"]
        + weights["location"]
    )

    if not np.isclose(total_weight, 1.0):
        raise ValueError("Scoring weights must sum to 1.0.")

    score = (
        weights["text"] * (text_similarity * 100)
        + weights["mbti"] * mbti_score
        + weights["location"] * location_score
    )

    return float(np.clip(score, 0, 100))


# Test the hybrid scoring engine
test_hybrid_score = calculate_compatibility_score(
    text_similarity=0.80,
    mbti_score=90,
    location_score=100
)

print("Test compatibility score:",
      round(test_hybrid_score, 2))

print("Initial weights:",
      INITIAL_WEIGHTS)

In [ ]:
# ============================================
# User-to-User Compatibility Matching Function
# ============================================

user_index = users_df.set_index("user_id")


def match_users(user_id_a, user_id_b, weights=None):
    """
    Calculate compatibility between two User IDs.

    Returns:
        Compatibility score from 0 to 100.
    """

    if user_id_a not in user_index.index:
        raise ValueError(f"Unknown User ID: {user_id_a}")

    if user_id_b not in user_index.index:
        raise ValueError(f"Unknown User ID: {user_id_b}")

    if user_id_a == user_id_b:
        raise ValueError("A user cannot be matched with themselves.")

    user_a = user_index.loc[user_id_a]
    user_b = user_index.loc[user_id_b]

    # Find the embedding positions
    index_a = users_df.index[
        users_df["user_id"] == user_id_a
    ][0]

    index_b = users_df.index[
        users_df["user_id"] == user_id_b
    ][0]

    # Semantic similarity
    text_similarity = float(
        text_similarity_matrix[index_a, index_b]
    )

    # MBTI compatibility
    mbti_score = get_mbti_score(
        user_a["mbti"],
        user_b["mbti"]
    )

    # Location compatibility
    location_score = get_location_score(
        user_a["location"],
        user_b["location"]
    )

    # Final hybrid score
    compatibility_score = calculate_compatibility_score(
        text_similarity=text_similarity,
        mbti_score=mbti_score,
        location_score=location_score,
        weights=weights
    )

    return round(compatibility_score, 2)


# Test the matching function
score = match_users("U001", "U002")

print("U001 ↔ U002")
print("Compatibility:", score, "%")

In [ ]:
# ============================================
# Adaptive Feedback Learning - Feature Preparation
# ============================================

feedback_features = []

for _, row in feedback_df.iterrows():

    user_id = row["user_id"]
    matched_user_id = row["matched_user_id"]

    user_a = user_index.loc[user_id]
    user_b = user_index.loc[matched_user_id]

    index_a = users_df.index[
        users_df["user_id"] == user_id
    ][0]

    index_b = users_df.index[
        users_df["user_id"] == matched_user_id
    ][0]

    # Semantic similarity feature
    text_similarity = float(
        text_similarity_matrix[index_a, index_b]
    )

    # MBTI compatibility feature
    mbti_score = get_mbti_score(
        user_a["mbti"],
        user_b["mbti"]
    )

    # Location compatibility feature
    location_score = get_location_score(
        user_a["location"],
        user_b["location"]
    )

    feedback_features.append({
        "user_id": user_id,
        "matched_user_id": matched_user_id,
        "text_similarity": text_similarity,
        "mbti_score": mbti_score / 100,
        "location_score": location_score / 100,
        "action": int(row["action"]),
        "timestamp": row["timestamp"]
    })

feedback_learning_df = pd.DataFrame(feedback_features)

print("Feedback learning dataset created.")
print("Rows:", len(feedback_learning_df))
print("Columns:")
print(feedback_learning_df.columns.tolist())

feedback_learning_df.head()

In [ ]:
# ============================================
# Adaptive Feedback Learning Model
# ============================================

from sklearn.linear_model import LogisticRegression

adaptive_models = {}

FEATURE_COLUMNS = [
    "text_similarity",
    "mbti_score",
    "location_score"
]

for user_id, user_feedback in feedback_learning_df.groupby("user_id"):

    X = user_feedback[FEATURE_COLUMNS]
    y = user_feedback["action"]

    # A model needs both Accept and Reject examples
    if y.nunique() < 2:
        continue

    model = LogisticRegression(
        random_state=42,
        max_iter=1000
    )

    model.fit(X, y)

    adaptive_models[user_id] = model

print("Adaptive models trained:", len(adaptive_models))
print("Total users:", users_df["user_id"].nunique())

In [ ]:
# ============================================
# Regenerate Feedback with Balanced User Behavior
# ============================================

from datetime import datetime, timedelta

random.seed(42)
np.random.seed(42)

feedback_records = []

user_records = users_df.to_dict("records")

for user in user_records:

    candidates = [
        other for other in user_records
        if other["user_id"] != user["user_id"]
    ]

    selected_matches = random.sample(
        candidates,
        INTERACTIONS_PER_USER
    )

    user_interactions = []

    for matched in selected_matches:

        same_location = (
            user["location"] == matched["location"]
        )

        same_profession = (
            user["profession"] == matched["profession"]
        )

        shared_interests = len(
            set(user["interests"].split(", "))
            &
            set(matched["interests"].split(", "))
        )

        same_mbti = (
            user["mbti"] == matched["mbti"]
        )

        preference_score = 0.20

        if same_location:
            preference_score += 0.15

        if same_profession:
            preference_score += 0.25

        if shared_interests >= 1:
            preference_score += 0.15

        if shared_interests >= 2:
            preference_score += 0.10

        if same_mbti:
            preference_score += 0.05

        preference_score += np.random.normal(0, 0.08)

        preference_score = np.clip(
            preference_score,
            0.05,
            0.95
        )

        action = int(
            np.random.random() < preference_score
        )

        user_interactions.append({
            "user_id": user["user_id"],
            "matched_user_id": matched["user_id"],
            "action": action
        })

    # Ensure both Accept and Reject examples exist
    if all(x["action"] == 1 for x in user_interactions):
        user_interactions[0]["action"] = 0

    elif all(x["action"] == 0 for x in user_interactions):
        user_interactions[0]["action"] = 1

    # Add timestamps
    for interaction in user_interactions:

        days_offset = random.randint(0, 90)

        timestamp = (
            datetime(2025, 1, 1)
            + timedelta(days=days_offset)
        ).strftime("%Y-%m-%d")

        interaction["timestamp"] = timestamp

        feedback_records.append(interaction)


feedback_df = pd.DataFrame(feedback_records)

print("Feedback rows:", len(feedback_df))
print("Users:", feedback_df["user_id"].nunique())
print("Accepts:", int(feedback_df["action"].sum()))
print("Rejects:", int((feedback_df["action"] == 0).sum()))

In [ ]:
# ============================================
# Validate Regenerated Feedback Dataset
# ============================================

interaction_counts = feedback_df["user_id"].value_counts()

user_action_counts = (
    feedback_df
    .groupby("user_id")["action"]
    .nunique()
)

print("Total rows:", len(feedback_df))
print("Unique users:", feedback_df["user_id"].nunique())

print("\nInteractions per user:")
print("Minimum:", interaction_counts.min())
print("Maximum:", interaction_counts.max())

print("\nUsers with both Accept and Reject:")
print((user_action_counts == 2).sum())

print("Users with only one action type:")
print((user_action_counts == 1).sum())

print("\nDuplicate rows:", feedback_df.duplicated().sum())
print("Invalid actions:",
      set(feedback_df["action"]) - {0, 1})

In [ ]:
# ============================================
# Adaptive Feedback Learning - Feature Preparation
# ============================================

feedback_features = []

for _, row in feedback_df.iterrows():

    user_id = row["user_id"]
    matched_user_id = row["matched_user_id"]

    user_a = user_index.loc[user_id]
    user_b = user_index.loc[matched_user_id]

    index_a = users_df.index[
        users_df["user_id"] == user_id
    ][0]

    index_b = users_df.index[
        users_df["user_id"] == matched_user_id
    ][0]

    text_similarity = float(
        text_similarity_matrix[index_a, index_b]
    )

    mbti_score = get_mbti_score(
        user_a["mbti"],
        user_b["mbti"]
    )

    location_score = get_location_score(
        user_a["location"],
        user_b["location"]
    )

    feedback_features.append({
        "user_id": user_id,
        "matched_user_id": matched_user_id,
        "text_similarity": text_similarity,
        "mbti_score": mbti_score / 100,
        "location_score": location_score / 100,
        "action": int(row["action"]),
        "timestamp": row["timestamp"]
    })

feedback_learning_df = pd.DataFrame(feedback_features)

print("Feedback learning dataset rebuilt.")
print("Rows:", len(feedback_learning_df))
print("Users:", feedback_learning_df["user_id"].nunique())

In [ ]:
# ============================================
# Adaptive Feedback Learning Model
# ============================================

from sklearn.linear_model import LogisticRegression

FEATURE_COLUMNS = [
    "text_similarity",
    "mbti_score",
    "location_score"
]

adaptive_models = {}

for user_id, user_feedback in feedback_learning_df.groupby("user_id"):

    X = user_feedback[FEATURE_COLUMNS]
    y = user_feedback["action"]

    model = LogisticRegression(
        random_state=42,
        max_iter=1000
    )

    model.fit(X, y)

    adaptive_models[user_id] = model

print("Adaptive models trained:", len(adaptive_models))
print("Total users:", users_df["user_id"].nunique())

In [ ]:
# ============================================
# Inspect Personalized Learned Weights
# ============================================

learned_weights = []

for user_id, model in adaptive_models.items():

    coefficients = model.coef_[0]

    learned_weights.append({
        "user_id": user_id,
        "text_coefficient": coefficients[0],
        "mbti_coefficient": coefficients[1],
        "location_coefficient": coefficients[2]
    })

learned_weights_df = pd.DataFrame(learned_weights)

print("Learned weights generated for:",
      len(learned_weights_df), "users")

print("\nSample learned coefficients:")
print(learned_weights_df.head())

print("\nCoefficient ranges:")
print(
    learned_weights_df[
        [
            "text_coefficient",
            "mbti_coefficient",
            "location_coefficient"
        ]
    ].describe()
)

In [ ]:
# ============================================
# Personalized Adaptive Scoring Weights
# ============================================

def coefficients_to_weights(coefficients):
    """
    Convert learned model coefficients into
    personalized non-negative weights that sum to 1.
    """

    coefficients = np.asarray(coefficients, dtype=float)

    # Numerical stability
    shifted = coefficients - np.max(coefficients)

    exp_values = np.exp(shifted)

    weights = exp_values / exp_values.sum()

    return weights


personalized_weights = {}

for user_id, model in adaptive_models.items():

    coefficients = model.coef_[0]

    weights = coefficients_to_weights(coefficients)

    personalized_weights[user_id] = {
        "text": float(weights[0]),
        "mbti": float(weights[1]),
        "location": float(weights[2])
    }


# Create a readable DataFrame
personalized_weights_df = pd.DataFrame([
    {
        "user_id": user_id,
        "text_weight": weights["text"],
        "mbti_weight": weights["mbti"],
        "location_weight": weights["location"]
    }
    for user_id, weights in personalized_weights.items()
])


print("Personalized weights generated:",
      len(personalized_weights_df))

print("\nSample personalized weights:")
print(personalized_weights_df.head())

print("\nWeight-sum check:")
print(
    personalized_weights_df[
        ["text_weight", "mbti_weight", "location_weight"]
    ].sum(axis=1).head()
)

In [ ]:
# ============================================
# Validate Personalized Adaptive Weights
# ============================================

weight_columns = [
    "text_weight",
    "mbti_weight",
    "location_weight"
]

weight_sums = personalized_weights_df[weight_columns].sum(axis=1)

print("Number of personalized users:",
      len(personalized_weights_df))

print(
    "Minimum weight sum:",
    round(weight_sums.min(), 6)
)

print(
    "Maximum weight sum:",
    round(weight_sums.max(), 6)
)

print("\nWeight ranges:")

for column in weight_columns:
    print(
        column,
        "→",
        round(personalized_weights_df[column].min(), 4),
        "to",
        round(personalized_weights_df[column].max(), 4)
    )

print("\nAll weights non-negative:",
      (personalized_weights_df[weight_columns] >= 0).all().all())

print(
    "All weight sums equal 1:",
    np.allclose(weight_sums, 1.0)
)

In [ ]:
# ============================================
# Adaptive Personalized Matching Function
# ============================================

def match_users_adaptive(user_id_a, user_id_b):
    """
    Calculate compatibility using personalized weights
    learned from the viewing user's feedback history.
    """

    if user_id_a not in user_index.index:
        raise ValueError(f"Unknown User ID: {user_id_a}")

    if user_id_b not in user_index.index:
        raise ValueError(f"Unknown User ID: {user_id_b}")

    if user_id_a == user_id_b:
        raise ValueError("A user cannot be matched with themselves.")

    user_a = user_index.loc[user_id_a]
    user_b = user_index.loc[user_id_b]

    index_a = users_df.index[
        users_df["user_id"] == user_id_a
    ][0]

    index_b = users_df.index[
        users_df["user_id"] == user_id_b
    ][0]

    text_similarity = float(
        text_similarity_matrix[index_a, index_b]
    )

    mbti_score = get_mbti_score(
        user_a["mbti"],
        user_b["mbti"]
    )

    location_score = get_location_score(
        user_a["location"],
        user_b["location"]
    )

    # Use personalized weights learned for the viewing user
    weights = personalized_weights.get(
        user_id_a,
        INITIAL_WEIGHTS
    )

    score = calculate_compatibility_score(
        text_similarity=text_similarity,
        mbti_score=mbti_score,
        location_score=location_score,
        weights=weights
    )

    return round(score, 2)


# Compare fixed-weight vs adaptive-weight scoring
user_a = "U001"
user_b = "U002"

fixed_score = match_users(
    user_a,
    user_b,
    weights=INITIAL_WEIGHTS
)

adaptive_score = match_users_adaptive(
    user_a,
    user_b
)

print(f"{user_a} ↔ {user_b}")
print("Initial-weight score:", fixed_score, "%")
print("Adaptive-weight score:", adaptive_score, "%")
print(
    "Personalized weights:",
    personalized_weights[user_a]
)

In [ ]:
# ============================================
# Top-5 Personalized Recommendation Engine
# ============================================

def get_top_5_matches(user_id):
    """
    Return the top 5 personalized matches for a user.
    """

    if user_id not in user_index.index:
        raise ValueError(f"Unknown User ID: {user_id}")

    recommendations = []

    for candidate_id in users_df["user_id"]:

        if candidate_id == user_id:
            continue

        score = match_users_adaptive(
            user_id,
            candidate_id
        )

        candidate = user_index.loc[candidate_id]

        recommendations.append({
            "user_id": candidate_id,
            "name": candidate["name"],
            "profession": candidate["profession"],
            "location": candidate["location"],
            "mbti": candidate["mbti"],
            "compatibility_score": score
        })

    recommendations_df = pd.DataFrame(recommendations)

    recommendations_df = (
        recommendations_df
        .sort_values(
            "compatibility_score",
            ascending=False
        )
        .head(5)
        .reset_index(drop=True)
    )

    recommendations_df.index += 1

    return recommendations_df


# Test the Top-5 recommendation engine
top_5 = get_top_5_matches("U001")

print("Top 5 matches for U001:")
display(top_5)

In [ ]:
# ============================================
# Save Project 1 Model Artifacts
# ============================================

import joblib
import os

ARTIFACTS_DIR = f"{PROJECT_ROOT}/models"
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

# Save Sentence Transformer model name
joblib.dump(
    EMBEDDING_MODEL_NAME,
    f"{ARTIFACTS_DIR}/embedding_model_name.joblib"
)

# Save MBTI compatibility rules
joblib.dump(
    MBTI_COMPATIBILITY,
    f"{ARTIFACTS_DIR}/mbti_compatibility.joblib"
)

# Save initial scoring weights
joblib.dump(
    INITIAL_WEIGHTS,
    f"{ARTIFACTS_DIR}/initial_weights.joblib"
)

# Save personalized adaptive weights
joblib.dump(
    personalized_weights,
    f"{ARTIFACTS_DIR}/personalized_weights.joblib"
)

# Save adaptive models
joblib.dump(
    adaptive_models,
    f"{ARTIFACTS_DIR}/adaptive_models.joblib"
)

print("Project 1 artifacts saved successfully.")
print("Artifacts directory:", ARTIFACTS_DIR)

In [ ]:
# ============================================
# Adaptive Recommendation Performance Analysis
# ============================================

from sklearn.metrics import precision_score, recall_score, f1_score

baseline_predictions = []
adaptive_predictions = []
actual_feedback = []

for _, row in feedback_df.iterrows():

    user_id = row["user_id"]
    matched_user_id = row["matched_user_id"]

    # Actual user feedback
    actual_feedback.append(int(row["action"]))

    # Baseline score
    baseline_score = match_users(
        user_id,
        matched_user_id,
        weights=INITIAL_WEIGHTS
    )

    # Adaptive score
    adaptive_score = match_users_adaptive(
        user_id,
        matched_user_id
    )

    # Convert compatibility score to predicted Accept/Reject
    baseline_predictions.append(
        int(baseline_score >= 50)
    )

    adaptive_predictions.append(
        int(adaptive_score >= 50)
    )


# Baseline metrics
baseline_precision = precision_score(
    actual_feedback,
    baseline_predictions,
    zero_division=0
)

baseline_recall = recall_score(
    actual_feedback,
    baseline_predictions,
    zero_division=0
)

baseline_f1 = f1_score(
    actual_feedback,
    baseline_predictions,
    zero_division=0
)


# Adaptive metrics
adaptive_precision = precision_score(
    actual_feedback,
    adaptive_predictions,
    zero_division=0
)

adaptive_recall = recall_score(
    actual_feedback,
    adaptive_predictions,
    zero_division=0
)

adaptive_f1 = f1_score(
    actual_feedback,
    adaptive_predictions,
    zero_division=0
)


performance_results = pd.DataFrame({
    "Metric": [
        "Precision",
        "Recall",
        "F1 Score"
    ],
    "Baseline": [
        baseline_precision,
        baseline_recall,
        baseline_f1
    ],
    "Adaptive": [
        adaptive_precision,
        adaptive_recall,
        adaptive_f1
    ]
})

performance_results["Improvement"] = (
    performance_results["Adaptive"]
    - performance_results["Baseline"]
)

print("Recommendation Performance Comparison")
display(performance_results)

In [ ]:
# ============================================
# Save Performance Analysis Results
# ============================================

PERFORMANCE_PATH = f"{PROJECT_ROOT}/outputs/performance_results.csv"

performance_results.to_csv(
    PERFORMANCE_PATH,
    index=False
)

print("Performance analysis saved successfully.")
print("Path:", PERFORMANCE_PATH)

In [ ]:
# ============================================
# Performance Comparison Visualization
# ============================================

import matplotlib.pyplot as plt

metrics = performance_results["Metric"]

baseline_values = performance_results["Baseline"]
adaptive_values = performance_results["Adaptive"]

x = np.arange(len(metrics))
width = 0.35

plt.figure(figsize=(9, 5))

plt.bar(
    x - width / 2,
    baseline_values,
    width,
    label="Baseline"
)

plt.bar(
    x + width / 2,
    adaptive_values,
    width,
    label="Adaptive"
)

plt.xticks(x, metrics)
plt.ylabel("Score")
plt.ylim(0, 1)
plt.title("Baseline vs Adaptive Recommendation Performance")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# Save Final Validated Datasets
# ============================================

# Save the final user dataset
users_df[USER_COLUMNS].to_csv(
    USERS_PATH,
    index=False
)

# Save the final regenerated feedback dataset
feedback_df[FEEDBACK_COLUMNS].to_csv(
    FEEDBACK_PATH,
    index=False
)

print("Final datasets saved successfully.")
print("users.csv rows:", len(users_df))
print("feedback.csv rows:", len(feedback_df))

In [ ]:
# ============================================
# Save NLP and Similarity Artifacts
# ============================================

import numpy as np

# Save Sentence Transformer embeddings
np.save(
    f"{ARTIFACTS_DIR}/profile_embeddings.npy",
    profile_embeddings
)

# Save cosine similarity matrix
np.save(
    f"{ARTIFACTS_DIR}/text_similarity_matrix.npy",
    text_similarity_matrix
)

print("NLP artifacts saved successfully.")
print("Embedding shape:", profile_embeddings.shape)
print("Similarity matrix shape:", text_similarity_matrix.shape)

In [ ]:
# ============================================
# Project 1 Model Summary
# ============================================

project_summary = {
    "project": "UNLOX Major Project 1",
    "title": "Profile-Based Matching Algorithm",
    "users": len(users_df),
    "feedback_records": len(feedback_df),
    "embedding_model": EMBEDDING_MODEL_NAME,
    "embedding_dimensions": int(profile_embeddings.shape[1]),
    "initial_weights": INITIAL_WEIGHTS,
    "adaptive_users": len(adaptive_models),
    "top_k_recommendations": 5,
    "baseline_precision": float(baseline_precision),
    "adaptive_precision": float(adaptive_precision),
    "baseline_recall": float(baseline_recall),
    "adaptive_recall": float(adaptive_recall),
    "baseline_f1": float(baseline_f1),
    "adaptive_f1": float(adaptive_f1)
}

project_summary_df = pd.DataFrame(
    list(project_summary.items()),
    columns=["parameter", "value"]
)

SUMMARY_PATH = f"{PROJECT_ROOT}/outputs/project_summary.csv"

project_summary_df.to_csv(
    SUMMARY_PATH,
    index=False
)

print("Project summary saved successfully.")
print("Path:", SUMMARY_PATH)

display(project_summary_df)

In [ ]:
# ============================================
# Project 1 - Final File Inventory
# ============================================

import os

print("PROJECT 1 FILES")
print("=" * 50)

for root, dirs, files in os.walk(PROJECT_ROOT):
    for file in files:
        relative_path = os.path.relpath(
            os.path.join(root, file),
            PROJECT_ROOT
        )
        print(relative_path)

In [ ]:
# ============================================
# Save Sentence Transformer Model
# ============================================

EMBEDDING_MODEL_PATH = f"{ARTIFACTS_DIR}/sentence_transformer_model"

embedding_model.save(
    EMBEDDING_MODEL_PATH
)

print("Sentence Transformer model saved successfully.")
print("Path:", EMBEDDING_MODEL_PATH)

In [ ]:
# ============================================
# Final Project 1 Artifact Verification
# ============================================

print("FINAL PROJECT 1 ARTIFACTS")
print("=" * 55)

for root, dirs, files in os.walk(PROJECT_ROOT):
    for file in files:
        relative_path = os.path.relpath(
            os.path.join(root, file),
            PROJECT_ROOT
        )
        print(relative_path)

In [ ]:
# ============================================
# Package Project 1 for VS Code
# ============================================

import shutil

PROJECT_ZIP = "/content/UNLOX_Major_Project_1_Final"

shutil.make_archive(
    PROJECT_ZIP,
    "zip",
    PROJECT_ROOT
)

print("Project 1 package created successfully.")
print(PROJECT_ZIP + ".zip")
